In [36]:
import pandas as pd

In [37]:
# def read_tbl():
#     df = pd.read_csv("../data/output/df.csv")
#     return df 


# df = read_tbl()

# df_new = df.copy()
# df_new = df_new.sample(n=100)

# # df_new.to_csv("../data/output/review.csv", index=False)

In [38]:
# def read_post():
#     df = pd.read_csv("../../preprocess/data/output/post.csv")
#     return df 

# post = read_post()

# post = post[post.person_nbr.fillna("").str.contains(r"B36-D90")]

# post

In [39]:
# df = pd.read_csv("data/output/df.csv")
# df = df[~((df.post_uid.fillna("") == ""))]
# # df = df[df.source_agency.str.lower().str.contains(r"district|redding|san fran|fullerton|orange|contra|riverside|siski|ontario")]
# # df.to_csv("data/output/review.csv", index=False)
# # df.to_csv("data/output/review.csv", index=False)

# df.to_csv("data/output/df.csv", index=False)

In [40]:
# df = pd.read_csv("data/output/df.csv")

# df = df[~(df.last_name.fillna("") == "")]

# df = df[~(df.first_name.fillna("") == "")]

# df.to_csv("data/output/df_clean.csv", index=False)

In [41]:
df = pd.read_csv("data/output/df.csv")
df = df[~((df.match_probability.fillna("") == ""))]

def analyze_probabilities(df):
    """Analyze the distribution of match_probability column"""
    probs = df['match_probability'].dropna()
    
    print(f"Total records: {len(probs)}")
    print(f"\nBasic stats:")
    print(f"  Mean: {probs.mean():.3f}")
    print(f"  Median: {probs.median():.3f}")
    print(f"  Std Dev: {probs.std():.3f}")
    print(f"  Min: {probs.min():.3f} | Max: {probs.max():.3f}")
    
    print(f"\nPercentiles:")
    for p in [25, 50, 75, 90, 95, 99]:
        print(f"  {p}th: {probs.quantile(p/100):.3f}")
    
    print(f"\nBinned distribution:")
    bins = [0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
    counts = pd.cut(probs, bins=bins, include_lowest=True).value_counts().sort_index()
    for bin_range, count in counts.items():
        pct = count / len(probs) * 100
        print(f"  {bin_range}: {count:,} ({pct:.1f}%)")

analyze_probabilities(df)



Total records: 661

Basic stats:
  Mean: 0.777
  Median: 0.811
  Std Dev: 0.074
  Min: 0.511 | Max: 0.811

Percentiles:
  25th: 0.805
  50th: 0.811
  75th: 0.811
  90th: 0.811
  95th: 0.811
  99th: 0.811

Binned distribution:
  (-0.001, 0.5]: 0 (0.0%)
  (0.5, 0.7]: 112 (16.9%)
  (0.7, 0.8]: 18 (2.7%)
  (0.8, 0.9]: 531 (80.3%)
  (0.9, 0.95]: 0 (0.0%)
  (0.95, 1.0]: 0 (0.0%)


In [42]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy import stats

def analyze_probabilities(df):
    """Analyze the distribution of match_probability column with visualizations"""
    probs = df['match_probability'].dropna()
    
    print("=" * 60)
    print("MATCH PROBABILITY ANALYSIS")
    print("=" * 60)
    print(f"\n📊 Total matches analyzed: {len(probs):,}")
    
    print(f"\n📈 SUMMARY STATISTICS")
    print(f"   Average probability: {probs.mean():.1%}")
    print(f"   Middle value (median): {probs.median():.1%}")
    print(f"   Spread (std dev): {probs.std():.3f}")
    print(f"   Range: {probs.min():.1%} to {probs.max():.1%}")
    
    print(f"\n🎯 KEY THRESHOLDS")
    print(f"   25% of matches are below: {probs.quantile(0.25):.1%}")
    print(f"   50% of matches are below: {probs.quantile(0.50):.1%}")
    print(f"   75% of matches are below: {probs.quantile(0.75):.1%}")
    print(f"   90% of matches are below: {probs.quantile(0.90):.1%}")
    print(f"   95% of matches are below: {probs.quantile(0.95):.1%}")
    
    print(f"\n📦 CONFIDENCE GROUPS")
    bins = [0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
    labels = ['Low (<50%)', 'Medium (50-70%)', 'Good (70-80%)', 
              'High (80-90%)', 'Very High (90-95%)', 'Excellent (95%+)']
    counts = pd.cut(probs, bins=bins, include_lowest=True).value_counts().sort_index()
    
    for (bin_range, count), label in zip(counts.items(), labels):
        pct = count / len(probs) * 100
        bar = '█' * int(pct / 2)  # Visual bar
        print(f"   {label:20s}: {count:6,} ({pct:5.1f}%) {bar}")
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            'Box Plot: What does the spread look like?', 
            'Distribution: How are probabilities clustered?'
        )
    )
    
    # Box and whisker plot
    fig.add_trace(
        go.Box(
            y=probs, 
            marker_color='lightblue',
            boxmean='sd',
            hovertemplate='<b>Value: %{y:.1%}</b><extra></extra>',
            showlegend=False  # Remove from legend
        ),
        row=1, col=1
    )
    
    # Simple histogram showing actual distribution
    fig.add_trace(
        go.Histogram(
            x=probs, 
            marker_color='steelblue', 
            opacity=0.8,
            nbinsx=30,
            hovertemplate='Probability: %{x:.1%}<br>Count: %{y}<extra></extra>',
            showlegend=False  # Remove from legend
        ),
        row=1, col=2
    )
    
    # Update layout
    fig.update_xaxes(title_text="Match Probability", tickformat='.0%', row=1, col=2)
    fig.update_yaxes(title_text="Match Probability", tickformat='.0%', row=1, col=1)
    fig.update_yaxes(title_text="Number of Matches", row=1, col=2)
    
    fig.update_layout(
        height=550,
        showlegend=False,  # Hide legend entirely since it's not needed
        title_text="<b>Match Probability Distribution Analysis</b><br>",
        title_x=0.5,
        font=dict(size=12),
        hovermode='closest'
    )
    
    fig.show()
    
    print("\n" + "=" * 60)

analyze_probabilities(df)

MATCH PROBABILITY ANALYSIS

📊 Total matches analyzed: 661

📈 SUMMARY STATISTICS
   Average probability: 77.7%
   Middle value (median): 81.1%
   Spread (std dev): 0.074
   Range: 51.1% to 81.1%

🎯 KEY THRESHOLDS
   25% of matches are below: 80.5%
   50% of matches are below: 81.1%
   75% of matches are below: 81.1%
   90% of matches are below: 81.1%
   95% of matches are below: 81.1%

📦 CONFIDENCE GROUPS
   Low (<50%)          :      0 (  0.0%) 
   Medium (50-70%)     :    112 ( 16.9%) ████████
   Good (70-80%)       :     18 (  2.7%) █
   High (80-90%)       :    531 ( 80.3%) ████████████████████████████████████████
   Very High (90-95%)  :      0 (  0.0%) 
   Excellent (95%+)    :      0 (  0.0%) 
